In [1]:
import pandas as pd

combined_df = pd.read_csv("dataset_preprocessed.csv")

In [2]:
combined_df.groupby("AI_flag")["review_comments"].describe()

,count,mean,std,min,25%,50%,75%,max
AI_flag,,,,,,,,
0,4561.0,2.228020,5.929832,0.0,0.0,0.0,2.0,115.0
1,4232.0,1.725425,6.013240,0.0,0.0,0.0,0.0,117.0


In [3]:
from scipy.stats import mannwhitneyu

ai = combined_df[combined_df["AI_flag"] == 1]["review_comments"]
human = combined_df[combined_df["AI_flag"] == 0]["review_comments"]

mannwhitneyu(ai, human)

MannwhitneyuResult(statistic=np.float64(8513963.5), pvalue=np.float64(6.376583101759101e-33))

In [4]:
combined_df["has_comments"] = combined_df["review_comments"] > 0

pd.crosstab(combined_df["has_comments"], combined_df["AI_flag"], normalize="columns")

AI_flag,0,1
has_comments,,
False,0.65205,0.774811
True,0.34795,0.225189


In [5]:
from scipy.stats import chi2_contingency

table = pd.crosstab(combined_df["has_comments"], combined_df["AI_flag"])

chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(160.44575373624662), pvalue=np.float64(9.041850791115065e-37), dof=1, expected_freq=array([[3243.48151939, 3009.51848061],
       [1317.51848061, 1222.48151939]]))

In [6]:
reviewed = combined_df[combined_df["review_comments"] > 0]

reviewed.groupby("AI_flag")["review_comments"].describe()

,count,mean,std,min,25%,50%,75%,max
AI_flag,,,,,,,,
0,1587.0,6.403277,8.622455,1.0,2.0,3.0,7.0,115.0
1,953.0,7.662120,10.731600,1.0,2.0,4.0,9.0,117.0


In [7]:
type_comments = combined_df.groupby(["pr_type", "AI_flag"])["review_comments"].mean().unstack()

type_comments["difference"] = type_comments[1] - type_comments[0]

type_comments

AI_flag,0,1,difference
pr_type,,,
bug_fix,2.570904,2.079715,-0.491189
documentation,2.193548,0.358779,-1.834770
feature,2.095801,2.476190,0.380390
maintenance,0.417840,0.036117,-0.381723
other,1.191693,3.450000,2.258307
refactor,1.110092,7.000000,5.889908
test,1.342857,0.310680,-1.032178


In [8]:
type_comments_reviewed = reviewed.groupby(["pr_type", "AI_flag"])["review_comments"].mean().unstack()

type_comments_reviewed["difference"] = type_comments_reviewed[1] - type_comments_reviewed[0]

type_comments_reviewed

AI_flag,0,1,difference
pr_type,,,
bug_fix,6.579846,7.865196,1.285350
documentation,6.181818,2.764706,-3.417112
feature,6.710084,7.333333,0.623249
maintenance,2.870968,2.666667,-0.204301
other,4.973333,6.900000,1.926667
refactor,4.481481,16.800000,12.318519
test,6.130435,4.571429,-1.559006


In [9]:
repo_review = combined_df.groupby(["repo", "AI_flag"])["has_comments"].mean().unstack()

repo_review["difference"] = repo_review[1] - repo_review[0]

repo_review

AI_flag,0,1,difference
repo,,,
antiwork/gumroad,0.376744,0.712389,0.335645
dotnet/aspnetcore,0.294737,0.037288,-0.257449
featureform/enrichmcp,0.225806,0.028571,-0.197235
glaredb/glaredb,0.335984,0.057047,-0.278937
jina-ai/node-deepresearch,0.000000,0.062500,0.062500
liam-hq/liam,0.622785,0.315789,-0.306995
microsoft/testfx,0.278333,0.082892,-0.195441
mlflow/mlflow,0.463284,0.614391,0.151107
pdfme/pdfme,0.349057,0.108303,-0.240753


In [10]:
(repo_review["difference"] > 0).sum(), len(repo_review)

(np.int64(4), 12)